# Qwen3 大模型指令微调实战——通过训练，让模型具备问答聊天的能力

以Qwen3作为基座大模型，通过参数微调的方式，实现垂直专业领域聊天，甚至支持DeepSeek R1 / QwQ式的带推理过程的对话，是学习LLM微调的入门任务。

## 使用Qwen进行微调

要想将Qwen3完成微调，我们要进行三个步骤：

1. 将模型加载进 NPU
2. 下载数据集并调整成正确格式
3. 运行模型训练


## 环境准备

本 Notebook 在 CANNLab 的 Ascend NPU 环境下运行。CANNLab 默认镜像已预装 `torch` 与 `torch_npu`，下方 cell 仅安装训练所需的上层依赖。

> **首次运行**：直接执行下方 cell 即可，安装完成后无需重启 kernel。
>
> **重跑场景**：如果环境里已经残留过旧版 `transformers` / `trl` 且本 Notebook 之前 import 过它们，安装后请点 `Kernel → Restart`，否则新版本不会生效。

In [ ]:
# 1. 安装训练所需依赖
%pip install -q -i https://pypi.tuna.tsinghua.edu.cn/simple modelscope==1.35.4 transformers==5.5.4 trl==1.2.0 datasets==4.8.4 swanlab==0.7.15 accelerate==1.13.0

# 2. 注册 Ascend NPU 后端：torch_npu 由 CANNLab 镜像预装，只需 import 即可让 PyTorch 识别 NPU
import torch
import torch_npu  # noqa: F401

print(f"torch: {torch.__version__}")
print(f"NPU 可用: {torch.npu.is_available()}, 卡数: {torch.npu.device_count()}")

## 下载并加载基座模型

首先我们需要从 ModelScope 中下载基座模型 Qwen3-0.6B-Base

In [ ]:
!modelscope download --model Qwen/Qwen3-0.6B-Base --local_dir ./Qwen3-0.6B-Base

接下来，我们可以将下载好的模型加载到 NPU 的显存中，同时，我们可以观察到 HBM 占用率的上升

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_PATH="./Qwen3-0.6B-Base"    # 模型路径

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    torch_dtype="auto",
    device_map="auto"
)

使用普通文本测试一下模型是否成功加载了

In [ ]:

# 输入的文本⬇️
text="Python 是一种"    # 普通文本——Base 模型擅长续写陈述性内容

# 模型推理代码⬇️
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
generation_ids = model.generate(**model_inputs,max_new_tokens=128)
content = tokenizer.decode(generation_ids[0], skip_special_tokens=True)
print(content)


使用聊天文本测试模型能力

> ⚠️ **预期结果会很怪**：当前模型是 `Qwen3-0.6B-Base`，**没有经过指令微调**，不知道如何按对话格式回答。下面这个 cell 的输出多半会是杂乱的续写或重复文字——这正是接下来我们要做 SFT 训练的原因。

In [ ]:
# 输入聊天文本⬇️
messages = [
    {"role": "user", "content": "什么是机器学习？"},
]

# 模型推理代码⬇️
text=tokenizer.apply_chat_template(messages,tokenize=False,add_generation_prompt=True,enable_thinking=False)
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
generation_ids = model.generate(**model_inputs,max_new_tokens=128)
content = tokenizer.decode(generation_ids[0], skip_special_tokens=True)
print(content)

---

## 加载我们要训练的数据集

Alpaca_zh 是面向中文场景的指令监督微调数据集，基于斯坦福大学 Alpaca 项目的核心结构设计，通过本地化改造适配中文语言习惯与任务需求。该数据集遵循 Alpaca 格式标准，每条数据包含必填的 instruction（任务指令）和 output（模型期望回答），以及可选的 input（任务输入）与 history（多轮对话历史）字段，其核心价值在于通过 指令-输入-输出 的强关联结构，优化大模型对中文任务的理解与执行能力。数据源包含从 Alpaca-3.5-en 翻译的 52K 指令样本，并融合中文社区优化的问答对，覆盖翻译、推理、摘要等场景。

数据集下载地址：

* Aplaca_zh数据集下载链接：https://www.modelscope.cn/datasets/llamafactory/alpaca_zh/summary

下载数据集命令

In [ ]:
!modelscope download --dataset llamafactory/alpaca_zh --local_dir ./alpaca_zh

### 读取数据集

In [ ]:
import datasets

alpaca_data = datasets.load_dataset("json", data_files="alpaca_zh/alpaca_data_zh_51k.json")

alpaca_data=alpaca_data["train"]
print(alpaca_data)

展示一条数据

In [ ]:
print(alpaca_data[0]["instruction"])
print(alpaca_data[0]["input"])
print(alpaca_data[0]["output"])

## 整理数据格式

将数据集拼接成我们之前模型能接受的聊天格式

In [ ]:
example=alpaca_data[0]

user_text = example["instruction"]+example["input"]
assistant_text = example["output"]
messages = [
    {"role": "user", "content": user_text},
    {"role": "assistant", "content": assistant_text},
]
text=tokenizer.apply_chat_template(messages,tokenize=False,add_generation_prompt=False)
print(text)

#### 通过函数对数据集进行批量处理

In [ ]:
def format_data(example):
    user_text = example["instruction"]+example["input"]
    assistant_text = example["output"]
    messages = [
        {"role": "user", "content": user_text},
        {"role": "assistant", "content": assistant_text},
    ]
    text=tokenizer.apply_chat_template(messages,tokenize=False,add_generation_prompt=False,enable_thinking=False)
    return {"text":text}

format_alpaca_data=alpaca_data.map(format_data,remove_columns=alpaca_data.column_names)

In [ ]:
print(format_alpaca_data[0]["text"])

### 切分数据集

In [ ]:
train_test_data=format_alpaca_data.train_test_split(0.01)
train_data = train_test_data["train"]
test_data = train_test_data["test"]

print("=====训练集数量")
print(train_data)
print("=====测试集数量")
print(test_data)

不使用lora微调⬇️ 

---

## 开始训练

### 构建训练参数

注意下方代码运行后，会在页面上方有一个输入框的弹窗出现，请在这里输入 SwanLab 的 API Key，然后按回车提交。可以从 https://swanlab.cn/settings 获取。如果不输入则会卡在这里无法继续运行。

In [ ]:
import getpass
import swanlab
from pathlib import Path
from trl import SFTConfig, SFTTrainer

# ====== SwanLab 项目配置 ======
# 下方的SWANLAB_PROJ请勿修改！
%env SWANLAB_PROJ=CANN_SwanLab

# 将下方的 SWANLAB_EXP_NAME 改成自己的名字，方便看出哪个实验是自己的！！支持中文
%env SWANLAB_EXP_NAME=Qwen3-Finetune-Instruction


# ====== SwanLab 登录（首次运行弹出安全输入框，凭证保存后下次自动复用） ======
def _swanlab_credential_saved() -> bool:
    """检查本地是否已有 SwanLab 凭证（默认存储在 ~/.netrc）"""
    netrc = Path.home() / ".netrc"
    if not netrc.exists():
        return False
    try:
        return "swanlab.cn" in netrc.read_text()
    except Exception:
        return False


if _swanlab_credential_saved():
    print("检测到本地已有 SwanLab 凭证，跳过登录。")
else:
    api_key = getpass.getpass("请输入你的 SwanLab API Key（从 https://swanlab.cn/settings 获取）: ")
    swanlab.login(api_key=api_key, save=True)
    print("SwanLab 登录成功！凭证已保存到 ~/.netrc，下次运行无需再输入。")


sft_config = SFTConfig(
    output_dir = "./sft_qwen",    # 保存位置
    max_steps=1000,    # 训练步长
    per_device_train_batch_size=4,    # 批量大小
    learning_rate = 0.00001,    # 学习率lr
    logging_steps=1,  # 打印日志的频率
    eval_strategy="steps",  # 开启评估
    eval_steps=0.2,   # 评估次数
    save_total_limit=2, # 设置最多保存的权重数量
    report_to="swanlab"   # 将日志报告给swanlab
)

### 开启训练

> ⏳ **训练耗时较长**（默认 1000 步约 30-60 分钟）。**点击运行后等当前 cell 跑完再点下一个 cell**，连续点会重复造成异常。SwanLab 训练曲线可在 [swanlab.cn](https://swanlab.cn) 实时查看。

In [ ]:
trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    args=sft_config,
    data_collator=None,
    train_dataset=train_data,
    eval_dataset=test_data,
    
)
trainer.train()
swanlab.finish()

---

## 保存模型

In [ ]:
model.save_pretrained(sft_config.output_dir, safe_serialization=True)
tokenizer.save_pretrained(sft_config.output_dir)

---

## 推理训练好的模型

In [ ]:
your_model_path="./sft_qwen"

tokenizer = AutoTokenizer.from_pretrained(your_model_path)
sft_model = AutoModelForCausalLM.from_pretrained(
    your_model_path,
    torch_dtype="auto",
    device_map="auto"
)

In [ ]:
messages = [
    {"role": "user", "content": "你好"},
]
text=tokenizer.apply_chat_template(messages,tokenize=False,add_generation_prompt=True,enable_thinking=False)
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
generation_ids = sft_model.generate(**model_inputs, max_new_tokens=128, eos_token_id=tokenizer.eos_token_id)
content = tokenizer.decode(
    generation_ids[0], skip_special_tokens=True
)
print(content)